# SpaceX Falcon 9 First Stage Landing Prediction

Reproducible analysis for the IBM Applied Data Science Capstone machine-learning lab. The notebook follows the course split (test size 0.2, random state 2) and compares logistic regression, SVM, a decision tree, and K-nearest neighbors.

**Data note:** the course has 90 launch records and 83 engineered features. The 18-row holdout makes each test example worth 5.6 percentage points, so treat accuracy as a small-sample estimate.


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score


In [1]:
DATA_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
FEATURES_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv'

data = pd.read_csv(DATA_URL)
X_raw = pd.read_csv(FEATURES_URL)
Y = data['Class'].to_numpy()

print(f'Rows: {len(data)}; engineered features: {X_raw.shape[1]}')
print('Class counts (0 = did not land, 1 = landed):')
print(pd.Series(Y).value_counts().sort_index().to_dict())


Rows: 90; engineered features: 83
Class counts (0 = did not land, 1 = landed):
{0: 30, 1: 60}


## Prepare the train and test sets

The course lab standardizes the full feature matrix before splitting. This is reproduced here for comparability. In production work, fit the scaler only on the training fold (inside a pipeline) to avoid leakage.


In [2]:
transform = StandardScaler()
X = transform.fit_transform(X_raw)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2
)
print('Train:', X_train.shape, 'Test:', X_test.shape)


Train: (72, 83) Test: (18, 83)


## Tune four classifiers with 10-fold cross-validation

The tree grid uses `max_features='sqrt'`. The supplied course notebook lists both `auto` and `sqrt`; in current scikit-learn versions `auto` is no longer accepted for decision trees, and for classifiers it had the same feature-selection meaning as `sqrt`.


In [3]:
model_specs = {
    'Logistic regression': (
        LogisticRegression(max_iter=2000),
        {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']},
    ),
    'SVM': (
        SVC(),
        {'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
         'C': np.logspace(-3, 3, 5),
         'gamma': np.logspace(-3, 3, 5)},
    ),
    'Decision tree': (
        DecisionTreeClassifier(random_state=0),
        {'criterion': ['gini', 'entropy'],
         'splitter': ['best', 'random'],
         'max_depth': [2*n for n in range(1, 10)],
         'max_features': ['sqrt'],
         'min_samples_leaf': [1, 2, 4],
         'min_samples_split': [2, 5, 10]},
    ),
    'K-nearest neighbors': (
        KNeighborsClassifier(),
        {'n_neighbors': list(range(1, 11)),
         'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
         'p': [1, 2]},
    ),
}

searches = {}
rows = []
for name, (estimator, grid) in model_specs.items():
    search = GridSearchCV(estimator, grid, cv=10, n_jobs=-1)
    search.fit(X_train, Y_train)
    searches[name] = search
    pred = search.predict(X_test)
    rows.append({
        'Model': name,
        'Best CV accuracy': search.best_score_,
        'Holdout accuracy': accuracy_score(Y_test, pred),
        'Balanced accuracy': balanced_accuracy_score(Y_test, pred),
        'Best parameters': search.best_params_,
    })

results = pd.DataFrame(rows).sort_values('Holdout accuracy', ascending=False)
results[['Model', 'Best CV accuracy', 'Holdout accuracy', 'Balanced accuracy']]


                 Model  Best CV accuracy  Holdout accuracy  Balanced accuracy
0  Logistic regression            0.8464            0.8333               0.75
1                  SVM            0.8482            0.8333               0.75
2        Decision tree            0.8625            0.8333               0.75
3  K-nearest neighbors            0.8482            0.8333               0.75

In [ ]:
for name, search in searches.items():
    pred = search.predict(X_test)
    print(f'{name}: best parameters={search.best_params_}')
    print(f'  CV accuracy={search.best_score_:.4f}; holdout accuracy={accuracy_score(Y_test, pred):.4f}')


## Logistic-regression confusion matrix

Rows are actual classes and columns are predicted classes in the order `[0, 1]`.


In [4]:
best_by_holdout = max(searches.items(), key=lambda item: accuracy_score(Y_test, item[1].predict(X_test)))
best_name, best_search = best_by_holdout
best_pred = best_search.predict(X_test)
print('Best holdout score:', best_name)
print('Confusion matrix [actual rows, predicted columns]:')
print(confusion_matrix(Y_test, best_pred, labels=[0, 1]))


Best holdout score: Logistic regression
Confusion matrix [actual rows, predicted columns]:
[[ 3  3]
 [ 0 12]]


## Findings and limits

- All four tuned models reached **83.3% holdout accuracy** (15 correct out of 18); the holdout score does not identify a unique winner.
- Logistic regression's confusion matrix is `[[3, 3], [0, 12]]`: 12 landed cases were correctly identified, 3 non-landings were predicted as landings, and no landed cases were missed.
- Balanced accuracy is **75.0%**, lower than raw accuracy because the test split has only six non-landing cases.
- The 90-record dataset is small. One changed test prediction shifts accuracy by 5.6 points. Validate on more launches and report uncertainty before using this model for operational or cost decisions.

The course lab describes reuse as a cost advantage, citing approximately $62 million for a Falcon 9 launch versus more than $165 million for other providers. That is background from the lab, not a cost estimate derived by this model.
